# Bab 1 - Latihan 1: Chatbot Serah Terima Shift

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-genai-migas/blob/main/colab/hari3/01-dasar-langchain-latihan-1.ipynb)

Ini versi **Google Colab**. Isi dan urutan contohnya sama dengan versi VS Code; yang berbeda hanya cell Persiapan di bawah, yang menyalin repository, memasang pustaka, dan mengisi API key di runtime Colab.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kode jawaban latihan ada di bagian **Kode Jawaban Latihan** pada [PDF bab ini](https://github.com/dteti-workshop/workshop-genai-migas/blob/main/hari3/01-dasar-langchain/01-dasar-langchain.pdf). Salin kodenya ke cell yang bertanda `TODO`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-genai-migas.git"
REPO = "/content/workshop-genai-migas" if os.path.isdir("/content") else os.path.abspath("workshop-genai-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

Isi bagian bertanda `TODO(...)` dengan kode dari bagian **Kode Jawaban Latihan** pada PDF bab ini, lalu jalankan sel satu per satu
dengan Shift+Enter. Baris `... = None  # TODO` atau `raise NotImplementedError` berarti kode itu harus Anda ganti.

Chatbot dengan riwayat percakapan: setiap giliran ditambahkan ke daftar pesan lalu seluruhnya dikirim ke model.

Bagian utama memakai dua giliran contoh. Sel terakhir membuka mode interaktif (ketik catatan sendiri)
setelah `INTERAKTIF` diubah menjadi `True`.

In [ ]:
# Langkah 1 - Menyiapkan model dan riwayat percakapan
import os

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

# TODO(B1.1): buat daftar messages berisi satu SystemMessage yang meminta asisten mencatat laporan operator, mengingatnya, dan menjawab ringkas dalam Bahasa Indonesia
messages = []  # TODO: ganti dengan implementasi Anda

print("Asisten: Halo, saya asisten serah terima shift. Sampaikan catatan Anda.")

## Langkah 2 - Fungsi satu giliran
Pesan pengguna ditambahkan ke riwayat, model dipanggil dengan seluruh riwayat, lalu jawabannya ikut disimpan.

In [ ]:
def kirim_giliran(masukan: str):
    """Satu giliran chat dengan riwayat percakapan."""
    # TODO(B1.2): tambahkan pesan pengguna (HumanMessage) ke riwayat messages
    pass  # TODO: ganti dengan implementasi Anda

    try:
        # TODO(B1.3): panggil model dengan seluruh riwayat messages
        response = None  # TODO: ganti dengan implementasi Anda
        print(f"\nAsisten: {response.content}")

        # TODO(B1.4): simpan jawaban model ke riwayat sebagai AIMessage agar giliran berikutnya bisa mengingatnya
        pass  # TODO: ganti dengan implementasi Anda
        print(f"Panjang percakapan: {len(messages)} pesan")

    except Exception as error:
        print(f"\nError: {error}")
        messages.pop()  # buang pesan pengguna yang gagal diproses agar riwayat tetap rapi

## Langkah 3 - Giliran pertama: melaporkan catatan

In [ ]:
masukan = "Shift pagi: separator V-SR1-101 level controller hunting, ESP SR-020 stabil."
print(f"\nAnda: {masukan}")
kirim_giliran(masukan)

## Langkah 4 - Giliran kedua: menanyakan ingatan
Asisten bisa menjawab karena seluruh riwayat dikirim ulang.

In [ ]:
masukan = "Apa saja yang tadi saya laporkan?"
print(f"\nAnda: {masukan}")
kirim_giliran(masukan)

print(f"\nSampai jumpa! Percakapan kita berisi {len(messages)} pesan.")

## Langkah 5 - Mencoba sendiri (opsional)
Ubah `INTERAKTIF` menjadi `True`, jalankan sel, lalu ketik catatan atau pertanyaan. Ketik `keluar` untuk berhenti.
Pertanyaan kosong dilewati.

In [ ]:
INTERAKTIF = False  # ubah ke True untuk mencoba sendiri

if INTERAKTIF:
    while True:
        try:
            masukan = input("\nAnda: ").strip()
        except (EOFError, KeyboardInterrupt):
            print(f"\n\nSampai jumpa! Percakapan kita berisi {len(messages)} pesan.")
            break

        if masukan.lower() in ("keluar", "exit", "quit"):
            print(f"\nSampai jumpa! Percakapan kita berisi {len(messages)} pesan.")
            break

        if not masukan:
            continue

        kirim_giliran(masukan)